In [1]:
from pathlib import Path
import re
from collections import defaultdict

import numpy as np
import pandas as pd

EXP = "debug"
N_LEVELS = 4
RIGID = "Rigid"
AFFINE = "Affine"
SYN = "SyN"
STAGES = [RIGID, AFFINE, SYN]

BOKEH_COLORBLIND = (
    "#0072B2",  # 0
    "#E69F00",  # 1
    "#F0E442",  # 2
    "#009E73",  # 3
    "#56B4E9",  # 4
    "#D55E00",  # 5
    "#CC79A7",  # 6
    "#000000",  # 7
)

PLOT_SUBJECTS = False
THRESHOLDS = [1e-2, 5e-3, 1e-3, 1e-4]

# Plotting utility


In [2]:
import plotly.graph_objects as go
from plotly.colors import hex_to_rgb
from plotly.subplots import make_subplots

STAGE_TO_EXP = {
    "Rigid": "0111",
    "Affine": "0011",
    "SyN": "0001",
}

STAGE_TO_EXP_NAME = {
    "Rigid": "01xx",
    "Affine": "001x",
    "SyN": "0001",
}

BOKEH_COLORBLIND = (
    "#0072B2",  # 0
    "#E69F00",  # 1
    # "#F0E442",  # 2
    "#009E73",  # 3
    # "#56B4E9",  # 4
    # "#D55E00",  # 5
    "#CC79A7",  # 6
    "#000000",  # 7
)

COLORS = {
    # "Rigid": BOKEH_COLORBLIND[3],
    # "Affine": BOKEH_COLORBLIND[6],
    # "SyN": BOKEH_COLORBLIND[0],
    "SyN (Moving)": BOKEH_COLORBLIND[0],
    "SyN (Fixed)": BOKEH_COLORBLIND[1],
}


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    r, g, b = hex_to_rgb(hex_color)
    return f"rgba({r}, {g}, {b}, {alpha})"


def add_hline(fig, y, text):
    # Only show the horizontal line in the last column
    line = dict(dash="solid", width=1, color="black")
    fig.add_hline(
        y,
        line=line,
    )
    fig.add_hline(
        y,
        col=N_LEVELS,
        line=line,
        annotation_text=text,
        annotation_position="right",
        annotation_xref="paper",
        annotation_font_size=12,
        annotation_showarrow=False,
        annotation_font_color="black",
    )


def get_area_ratio_under(
    *,
    actual: np.array,
    estimated: np.array,
) -> tuple[float, float]:
    area_under = np.sum(np.maximum(actual - estimated, 0))
    total_area = np.sum(actual)

    ratio_under = area_under / total_area if total_area > 0 else 0

    return ratio_under


def get_area_ratio_over(
    *,
    actual: np.array,
    estimated: np.array,
) -> tuple[float, float]:
    area_over = np.sum(np.maximum(estimated - actual, 0))
    total_area = np.sum(actual)

    ratio_over = area_over / total_area if total_area > 0 else 0

    return ratio_over


def show_area_ratio(
    fig: go.Figure,
    *,
    threshold: float,
    color: str,
    ratio_over: float,
    ratio_under: float,
    row: int,
    col: int,
    x: float = 0,
):
    text = (
        # " <br>"
        f"<b>T</b>: {threshold:.0e}<br>"
        f"<b>U</b>: {ratio_under * 100:.1f}<br>"
        f"<b>O</b>: {ratio_over * 100:.1f}<br>"
    )

    fig.add_annotation(
        x=x,
        y=1,
        row=row,
        col=col,
        text=text,
        showarrow=False,
        align="left",
        xref="x domain",
        yref="y domain",
        yanchor="top",
        xanchor="left",
        font=dict(size=12),
        font_color=color,
        bgcolor="white",
    )


def show_area_ratio_title(
    fig: go.Figure,
    *,
    row: int,
    col: int,
):
    title = "<b>Area Ratio %</b>"
    fig.add_annotation(
        x=0,
        y=1,
        row=row,
        col=col,
        text=title,
        showarrow=False,
        align="left",
        xref="x domain",
        yref="y domain",
        yanchor="top",
        xanchor="left",
        font=dict(size=12),
        font_color="black",
        bgcolor="white",
    )


def make_legend(fig: go.Figure):
    fig.add_trace(
        go.Scatter(
            x=[1],
            y=[0],
            name="vprec",
            line=dict(dash="solid", color="black"),
            mode="lines",
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=[1],
            y=[0],
            name="predicted",
            line=dict(dash="dash", color="black"),
            mode="lines",
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )


def adjust_layout(fig: go.Figure, title: str):
    fig.update_layout(
        height=len(STAGES) * 400,
        width=1200,
        font=dict(
            size=24,
            color="black",
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.025,  # position above the subplots
            xanchor="left",
            x=0,
            traceorder="normal",
        ),
        title=title,
        margin=dict(l=10, r=40, t=150, b=10),
        plot_bgcolor="white",
    )
    for i in range(1, len(STAGES) * N_LEVELS + 1):
        fig.update_layout(
            {
                f"xaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
                f"yaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
            }
        )
    fig.update_yaxes(rangemode="tozero")


def display_hardware_support(fig: go.Figure, hw_support: dict | None = None):
    # Only add this ONCE (e.g., at subplot [1,1] or outside the loop)
    fig.add_trace(
        go.Scatter(
            x=[1],
            y=[0],
            mode="lines",
            line=dict(dash="solid", color="lightgray"),
            name="hardware support",
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

    for k, v in hw_support.items():
        add_hline(fig, y=v, text=k)


def show_and_save(fig: go.Figure, out_file: Path | None, show: bool):
    if show:
        fig.show()
    if out_file is not None:
        fig.update_layout(
            title=None,
            margin=dict(t=10),
        )
        out_file.parent.mkdir(parents=True, exist_ok=True)
        fig.write_image(str(out_file), scale=2)


def viz_pmin(
    df: pd.DataFrame,
    *,
    vprec: str,
    thresholds: list[float],
    estimated: str,
    title: str,
    out_file: Path | None = None,
    show: bool = True,
):
    levels: list[int] = sorted(df["level"].unique())

    # Subplots
    fig = make_subplots(
        rows=len(STAGES),
        cols=N_LEVELS,
        subplot_titles=[
            f"[{STAGE_TO_EXP_NAME[stage]}] {stage}: Level {level}"
            for stage in STAGES
            for level in levels
        ],
        shared_xaxes=False,
        shared_yaxes=True,
        horizontal_spacing=0.025,
        vertical_spacing=0.1,
    )
    # Update subplot titles font size
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=18)

    # Map stage to row index
    stage_to_row = {stage: i + 1 for i, stage in enumerate(STAGES)}
    level_to_col = {level: i + 1 for i, level in enumerate(levels)}

    # Add traces
    ratios = defaultdict(lambda: defaultdict(list))

    for i, stage in enumerate(STAGES):
        for _, level in enumerate(levels):
            level_df = df[
                (df["stage"] == stage)
                & (df["level"] == level)
                & (df["exp_id"] == STAGE_TO_EXP[stage])
            ].copy()

            # Cut off iterations to the minimum max iteration across thresholds
            # Otherwise, some thresholds may have more iterations than others,
            # leading to misleading plots.
            x_max = level_df.groupby(["threshold"])["iteration"].max().min()
            level_df = level_df[level_df["iteration"] <= x_max]

            row = stage_to_row[stage]
            col = level_to_col[level]
            iterations = sorted(level_df["iteration"].unique())

            mean_estimate = level_df.groupby("iteration")[estimated].mean()
            fig.add_trace(
                go.Scatter(
                    x=iterations,
                    y=mean_estimate,
                    line=dict(
                        color="black",
                        width=3,
                        dash="dot",
                    ),
                    mode="lines",
                    name=stage,
                    showlegend=False,
                ),
                row=row,
                col=col,
            )

            for i, threshold in enumerate(thresholds):
                filtered_values = level_df[level_df["threshold"] == threshold]
                mean_values = filtered_values.groupby("iteration")[
                    [estimated, vprec]
                ].mean()

                fig.add_trace(
                    go.Scatter(
                        x=iterations,
                        y=mean_values[vprec],
                        line=dict(
                            color=hex_to_rgba(
                                BOKEH_COLORBLIND[
                                    thresholds.index(threshold) % len(BOKEH_COLORBLIND)
                                ],
                                alpha=0.6,
                            ),
                            width=3,
                            dash="solid",
                        ),
                        mode="lines",
                        name=stage,
                        showlegend=False,
                    ),
                    row=row,
                    col=col,
                )
                # Display area ratio statistics at each stage-level subplot
                ratio_under = get_area_ratio_under(
                    actual=mean_values[vprec], estimated=mean_values[estimated]
                )
                ratio_over = get_area_ratio_over(
                    actual=mean_values[vprec], estimated=mean_values[estimated]
                )

                ratios[threshold]["under"].append(ratio_under)
                ratios[threshold]["over"].append(ratio_over)

                show_area_ratio(
                    fig,
                    threshold=threshold,
                    color=BOKEH_COLORBLIND[i % len(BOKEH_COLORBLIND)],
                    ratio_over=ratio_over,
                    ratio_under=ratio_under,
                    row=row,
                    col=col,
                    x=i / len(thresholds),
                )
            # show_area_ratio_title(fig, row=row, col=col)

    # Summarize ratios
    for threshold, values in ratios.items():
        avg_under = np.nanmean(values["under"]) * 100
        std_under = np.nanstd(values["under"]) * 100
        avg_over = np.nanmean(values["over"]) * 100
        std_over = np.nanstd(values["over"]) * 100
        print(
            f"{threshold=:.0e},{avg_under=:0.2f}, {std_under=:0.2f}, {avg_over=:0.2f}, {std_over=:0.2f}"
        )

    make_legend(fig)
    hw_support = {"FP32": 24, "FP24": 16, "TF32": 11, "BF16": 8}
    # hw_support = {"FP64": 53, "FP32": 24, "FP24": 16, "TF32": 11, "BF16": 8}
    display_hardware_support(fig, hw_support=hw_support)
    adjust_layout(fig, title=title)

    # x-axis title
    fig.add_annotation(
        text="nth Iteration",
        xref="paper",
        yref="paper",
        x=0.5,
        y=-0.075,
        showarrow=False,
        font=dict(size=24),
    )
    # y-axis title
    fig.add_annotation(
        text="Number of bits",
        xref="paper",
        yref="paper",
        x=-0.075,
        y=0.5,
        textangle=-90,
        showarrow=False,
        font=dict(size=24),
    )
    fig.update_layout(margin=dict(b=80, l=80))

    show_and_save(fig, out_file, show)


# Test metrics


In [3]:
N = 10_000

actual = np.full(N, 5)
estimated = np.linspace(0, 10, N)
under = get_area_ratio_under(actual=actual, estimated=estimated)
over = get_area_ratio_over(actual=actual, estimated=estimated)
np.testing.assert_almost_equal(under, 0.25, decimal=4)
np.testing.assert_almost_equal(over, 0.25, decimal=4)

actual = np.full(N, 5)
estimated = np.full(N, 0)
under = get_area_ratio_under(actual=actual, estimated=estimated)
over = get_area_ratio_over(actual=actual, estimated=estimated)
np.testing.assert_almost_equal(under, 1, decimal=4)
np.testing.assert_almost_equal(over, 0, decimal=4)

actual = np.full(N, 5)
estimated = np.full(N, 10)
under = get_area_ratio_under(actual=actual, estimated=estimated)
over = get_area_ratio_over(actual=actual, estimated=estimated)
np.testing.assert_almost_equal(under, 0, decimal=4)
np.testing.assert_almost_equal(over, 1, decimal=4)

actual = np.full(N, 5)
estimated = np.full(N, 15)
under = get_area_ratio_under(actual=actual, estimated=estimated)
over = get_area_ratio_over(actual=actual, estimated=estimated)
np.testing.assert_almost_equal(under, 0, decimal=4)
np.testing.assert_almost_equal(over, 2, decimal=4)


actual = np.full(N, 5)
estimated = np.full(N, 5)
under = get_area_ratio_under(actual=actual, estimated=estimated)
over = get_area_ratio_over(actual=actual, estimated=estimated)
np.testing.assert_almost_equal(under, 0, decimal=4)
np.testing.assert_almost_equal(over, 0, decimal=4)

print("All tests passed!")

All tests passed!


# Data Ingest


## $P_{min}$


In [4]:
logs = Path("logs", "pmin").glob("*.log")
experiments = dict()
for log_path in logs:
    subject_id = log_path.stem
    experiments[subject_id] = log_path

In [5]:
p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST.*|  Elapsed time"

dfs = list()
for subject_id, log_path in experiments.items():
    log = log_path.read_text()
    raw_data = [
        [y.strip() for y in x.split("\n") if y.strip() not in ["", "XX"]]
        for i, x in enumerate(re.split(p_lvl_header, log))
        if i % 5 != 0
    ]
    for stage in STAGES:
        for level in range(N_LEVELS):
            table = defaultdict(list)
            for line in raw_data[level + N_LEVELS * STAGES.index(stage)]:
                if not (
                    line.startswith("2DIAGNOSTIC") or line.startswith("1DIAGNOSTIC")
                ):
                    # print(f"Skipping line: {line}")
                    continue

                cols = line.split(",")
                table["iteration"].append(int(cols[1]))
                table["metricValue"].append(np.float64(cols[2]))

                if line.startswith("2DIAGNOSTIC"):  # Rigid + Affine
                    table["lipschitz"].append(np.float64(cols[6]))
                    table["params_2norm"].append(np.float64(cols[7]))
                    table["grad_2norm"].append(np.float64(cols[8]))

                elif line.startswith("1DIAGNOSTIC"):  # SyN
                    # Moving
                    table["moving_lipschitz"].append(np.float64(cols[6]))
                    table["moving_params_2norm"].append(np.float64(cols[7]))
                    table["moving_grad_2norm"].append(np.float64(cols[8]))
                    # Fixed
                    table["fixed_lipschitz"].append(np.float64(cols[9]))
                    table["fixed_params_2norm"].append(np.float64(cols[10]))
                    table["fixed_grad_2norm"].append(np.float64(cols[11]))

            common_data = {
                "exp": EXP,
                "stage": stage,
                "level": level,
                "subject": subject_id,
                "iteration": table["iteration"],
                "metricValue": table["metricValue"],
            }

            if line.startswith("2DIAGNOSTIC"):  # Rigid + Affine
                dfs.append(
                    pd.DataFrame(
                        data=common_data
                        | {
                            "lipschitz": table["lipschitz"],
                            "params_2norm": table["params_2norm"],
                            "grad_2norm": table["grad_2norm"],
                        }
                    )
                )
            elif line.startswith("1DIAGNOSTIC"):  # SyN
                # Moving
                dfs.append(
                    pd.DataFrame(
                        data=common_data
                        | {
                            "direction": "moving",
                            "lipschitz": table["moving_lipschitz"],
                            "params_2norm": table["moving_params_2norm"],
                            "grad_2norm": table["moving_grad_2norm"],
                        }
                    )
                )
                # Fixed
                dfs.append(
                    pd.DataFrame(
                        data=common_data
                        | {
                            "direction": "fixed",
                            "lipschitz": table["fixed_lipschitz"],
                            "params_2norm": table["fixed_params_2norm"],
                            "grad_2norm": table["fixed_grad_2norm"],
                        }
                    )
                )

pmin_df = pd.concat(dfs, ignore_index=True)
pmin_df = pmin_df.astype(
    {
        "exp": str,
        "stage": str,
        "level": int,
        "iteration": int,
        "subject": str,
        "direction": str,
        "lipschitz": np.float64,
        "params_2norm": np.float64,
        "grad_2norm": np.float64,
        "metricValue": np.float64,
    }
)
pmin_df["pmin"] = np.log2(
    (4 + 3 * np.sqrt(2))
    * pmin_df["lipschitz"]
    * pmin_df["params_2norm"]
    / pmin_df["grad_2norm"]
)

## VPREC


In [6]:
PLOT_BOUNDARIES = False

In [7]:
from pathlib import Path
import re

DEBUG = True
N_SUBJECT = 50

FILENAME_PATTERN = re.compile(
    r"(?P<subject_id>\d+)-r(?P<range>\d+)-p(?P<precision>\d+)\.log"
)

figures = Path("paper", "figures")
figures.mkdir(exist_ok=True, parents=True)

In [8]:
from typing import Iterable


def get_failed_exec(logs: Path | Iterable[Path]) -> int:
    if isinstance(logs, Path):
        logs = [logs]

    failed = list()
    for log in logs:
        if log.read_text().count("Elapsed time (stage 0):") != 4:
            failed.append(log)
    return failed


In [9]:
# Group logs
import re
from collections import defaultdict


logs = list(Path("logs", "vprec-space_search").rglob("*.log"))

experiements = defaultdict(list)
for log in logs:
    if "binary64" in log.name:
        exp_id = f"{log.parent.name}-binary64"
    else:
        m = FILENAME_PATTERN.match(log.name)
        if not m:
            print(log)
            raise ValueError("Invalid log file name")
        exp_id = f"{log.parent.name}-r{m.group('range')}-p{m.group('precision')}"

    experiements[exp_id].append(log)


In [10]:
# Count failed execution for each range-precision pair
n_failed = defaultdict(dict)
for task_id, logs in experiements.items():
    failed = get_failed_exec(logs)

    if "binary64" in task_id:
        continue
    range_ = int(FILENAME_PATTERN.match(logs[0].name).group("range"))
    precision_ = int(FILENAME_PATTERN.match(logs[0].name).group("precision"))
    n_failed[range_][precision_] = len(failed)

    if len(failed) > 0:
        print(f"Task {task_id} (r{range_}-p{precision_}): {len(failed)} failed")
    if DEBUG and len(failed) != len(logs):
        for log in failed:
            print(log, end="\n\n")

if PLOT_BOUNDARIES:
    # Manually add failure for range 6
    n_failed[6] = {r: N_SUBJECT for r in range(7, 24)}
else:
    del n_failed[8][6]
    del n_failed[7][6]


Task 0011-r7-p6 (r7-p6): 50 failed
Task 0011-r8-p6 (r8-p6): 50 failed
Task 0100-r7-p6 (r7-p6): 50 failed
Task 0100-r8-p6 (r8-p6): 50 failed
Task 0111-r7-p6 (r7-p6): 50 failed
Task 0111-r8-p6 (r8-p6): 50 failed
Task 0110-r7-p6 (r7-p6): 50 failed
Task 0110-r8-p6 (r8-p6): 50 failed
Task 1111-r7-p6 (r7-p6): 50 failed
Task 1111-r8-p6 (r8-p6): 50 failed
Task 0001-r7-p6 (r7-p6): 50 failed
Task 0001-r8-p6 (r8-p6): 50 failed


In [11]:
import pandas as pd
import numpy as np


p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST|  Elapsed time"

dfs = list()

for exp_id, logs in experiements.items():
    for filename in logs:
        # Extract subject_id from log
        subject_id = filename.stem.split("-")[0]

        # Filter out the header from the stages
        txt = filename.read_text()
        all_data = [
            x
            for x in re.split(p_lvl_header, txt)[1:]
            if not x.startswith(" (stage 0):")
        ]

        # 3 stages with 4 levels of resolution each
        for stage_id, stage in enumerate(STAGES, 1):
            for i, level in enumerate(all_data[4 * (stage_id - 1) : 4 * stage_id]):
                table = defaultdict(list)
                for row in re.split(r"\n", level.strip("XX").strip()):
                    # Skip invalid rows
                    # e.g. error messages or write volumes to disk
                    if not ("2DIAGNOSTIC" in row or "1DIAGNOSTIC" in row):
                        continue

                    cols = row.split(",")
                    table["iteration"].append(int(cols[1]))
                    table["metric"].append(np.float64(cols[2]))
                    table["convergence_value"].append(np.float64(cols[3]))
                    table["total_time"].append(np.float64(cols[4]))
                    table["since_last"].append(np.float64(cols[5]))

                data_format = "-".join(exp_id.split("-")[1:])
                if data_format == "binary64":
                    precision = 53
                    range_ = 11
                else:
                    precision = int(data_format.split("-")[1].removeprefix("p"))
                    range_ = int(data_format.split("-")[0].removeprefix("r"))

                dfs.append(
                    pd.DataFrame(
                        data={
                            "subject": subject_id,
                            "exp_id": exp_id.split("-")[0],
                            "data_format": data_format,
                            "precision": precision,
                            "range": range_,
                            "stage": stage,
                            "level": i,
                            "iteration": table["iteration"],
                            "metric": table["metric"],
                            "convergence_value": table["convergence_value"],
                            "total_time": table["total_time"],
                            "since_last": table["since_last"],
                        }
                    )
                )

vprec_df = pd.concat(dfs, ignore_index=True)
vprec_df = vprec_df.astype(
    {
        "stage": str,
        "level": int,
        "iteration": int,
        "metric": np.float64,
        "convergence_value": np.float64,
        "total_time": np.float64,
        "since_last": np.float64,
    }
)

In [12]:
def error_on_duplicate_group(df: pd.DataFrame, *, cols: list[str]):
    is_duplicate = df.duplicated(cols, keep=False)
    if is_duplicate.any():
        raise ValueError("Duplicate entries found")


# Expected failure test
# Multiple experiments can have the same subject, stage, level, iteration
try:
    error_on_duplicate_group(
        vprec_df,
        cols=["subject", "stage", "level", "iteration"],
    )
except ValueError:
    print("Test passed: duplicate entries found as expected")

Test passed: duplicate entries found as expected


In [13]:
# Ensure no duplicates within binary64 data format

error_on_duplicate_group(
    vprec_df[vprec_df["data_format"] == "binary64"],
    cols=["subject", "stage", "level", "iteration"],
)
print("Test passed: no duplicate entries found for binary64 data format")

Test passed: no duplicate entries found for binary64 data format


## Merging $P_{min}$ and VPREC to compute diff $p_{min}$ at each iteration (Threshold at each iteration)


In [14]:
fig_dir_per_iter = Path("paper", "figures", "pmin-threshold-each-iteration")
fig_dir_per_iter.mkdir(exist_ok=True, parents=True)

In [15]:
# Step 1: extract the Binary64 metric per group
vprec_df_ = vprec_df.copy()

ref = (
    vprec_df_[vprec_df_["data_format"] == "binary64"]
    .groupby(["subject", "stage", "level", "iteration"])["metric"]
    .first()  # Only 1 element per group
    .rename("binary64_metric")
    .reset_index()
)
# Step 2: join back to the original DataFrame
vprec_df_ = vprec_df_.merge(
    ref, on=["subject", "stage", "level", "iteration"], how="left"
)
# Step 3: compute the difference
vprec_df_["metric_rel_diff"] = (
    vprec_df_["metric"] - vprec_df_["binary64_metric"]
) / np.abs(vprec_df_["binary64_metric"])


### Global


In [16]:
# VPREC minimum precision per stage, level, iteration
vprec = (
    vprec_df_.groupby(["exp_id", "precision", "range", "stage", "level", "iteration"])[
        "metric_rel_diff"
    ]
    .mean()
    .rename("mean_rel_diff")
    .reset_index()
)


# PMIN mean estimate per stage, level, iteration, direction
estimated_pmin = (
    pmin_df.groupby(["stage", "level", "iteration", "direction"])["pmin"]
    .mean()
    .rename("estimated_pmin")
    .reset_index()
)


def get_threshold_data(
    *, vprec: pd.DataFrame, estimated_pmin: pd.DataFrame, threshold: float
) -> pd.DataFrame:
    vprec = vprec[vprec["mean_rel_diff"] <= threshold]

    vprec_pmin = (
        vprec.groupby(["exp_id", "stage", "level", "iteration"])["precision"]
        .min()
        .rename("vprec_pmin")
        .reset_index()
    )

    plot_data = estimated_pmin.merge(
        vprec_pmin, on=["stage", "level", "iteration"], how="left"
    )
    plot_data["diff_pmin"] = plot_data["vprec_pmin"] - plot_data["estimated_pmin"]
    plot_data["threshold"] = threshold
    return plot_data


dfs = list()
for threshold in THRESHOLDS:
    plot_data = get_threshold_data(
        vprec=vprec, estimated_pmin=estimated_pmin, threshold=threshold
    )
    dfs.append(plot_data)

plot_data_per_iter = pd.concat(dfs, ignore_index=True)
plot_data_per_iter

,stage,level,iteration,direction,estimated_pmin,exp_id,vprec_pmin,diff_pmin,threshold
0,Affine,0,1,nan,8.228812,0000,53,44.771188,0.0100
1,Affine,0,1,nan,8.228812,0001,6,-2.228812,0.0100
2,Affine,0,1,nan,8.228812,0011,7,-1.228812,0.0100
3,Affine,0,1,nan,8.228812,0100,8,-0.228812,0.0100
4,Affine,0,1,nan,8.228812,0110,7,-1.228812,0.0100
...,...,...,...,...,...,...,...,...,...
25440,SyN,3,20,moving,12.538473,0011,15,2.461527,0.0001
25441,SyN,3,20,moving,12.538473,0100,7,-5.538473,0.0001
25442,SyN,3,20,moving,12.538473,0110,7,-5.538473,0.0001
25443,SyN,3,20,moving,12.538473,0111,15,2.461527,0.0001


In [17]:
viz_pmin(
    plot_data_per_iter,
    estimated="estimated_pmin",
    vprec="vprec_pmin",
    thresholds=THRESHOLDS,
    title="pmin at each iteration",
    out_file=fig_dir_per_iter / "pmin-overview.pdf",
)


threshold=1e-02,avg_under=7.71, std_under=7.35, avg_over=40.41, std_over=35.65
threshold=5e-03,avg_under=8.96, std_under=8.26, avg_over=34.77, std_over=30.91
threshold=1e-03,avg_under=12.52, std_under=9.82, avg_over=27.66, std_over=25.98
threshold=1e-04,avg_under=15.80, std_under=10.61, avg_over=21.52, std_over=17.90


### Per subject-level


In [18]:
# VPREC minimum precision per stage, level, iteration
subjects_vprec_means = vprec_df_.copy()

# PMIN mean estimate per stage, level, iteration, direction
subjects_pmin_means = (
    pmin_df.groupby(["subject", "stage", "level", "iteration", "direction"])["pmin"]
    .mean()
    .rename("estimated_pmin")
    .reset_index()
)


def get_threshold_data(
    *, vprec: pd.DataFrame, estimated_pmin: pd.DataFrame, threshold: float
) -> pd.DataFrame:
    vprec = vprec[vprec["metric_rel_diff"] <= threshold]

    vprec_pmin = (
        vprec.groupby(["subject", "exp_id", "stage", "level", "iteration"])["precision"]
        .min()
        .rename("vprec_pmin")
        .reset_index()
    )

    plot_data = estimated_pmin.merge(
        vprec_pmin, on=["subject", "stage", "level", "iteration"], how="left"
    )
    plot_data["diff_pmin"] = plot_data["vprec_pmin"] - plot_data["estimated_pmin"]
    plot_data["threshold"] = threshold
    return plot_data


dfs = list()
for threshold in THRESHOLDS:
    plot_data = get_threshold_data(
        vprec=subjects_vprec_means,
        estimated_pmin=subjects_pmin_means,
        threshold=threshold,
    )
    plot_data["threshold"] = threshold
    dfs.append(plot_data)
subjects_plot_data_per_iter = pd.concat(dfs, ignore_index=True)
subjects_plot_data_per_iter

,subject,stage,level,iteration,direction,estimated_pmin,exp_id,vprec_pmin,diff_pmin,threshold
0,0003001,Affine,0,1,nan,7.689411,0000,53.0,45.310589,0.0100
1,0003001,Affine,0,1,nan,7.689411,0001,6.0,-1.689411,0.0100
2,0003001,Affine,0,1,nan,7.689411,0011,7.0,-0.689411,0.0100
3,0003001,Affine,0,1,nan,7.689411,0100,7.0,-0.689411,0.0100
4,0003001,Affine,0,1,nan,7.689411,0110,7.0,-0.689411,0.0100
...,...,...,...,...,...,...,...,...,...,...
842721,0003064,SyN,3,20,moving,12.929560,0011,16.0,3.070440,0.0001
842722,0003064,SyN,3,20,moving,12.929560,0100,7.0,-5.929560,0.0001
842723,0003064,SyN,3,20,moving,12.929560,0110,9.0,-3.929560,0.0001
842724,0003064,SyN,3,20,moving,12.929560,0111,18.0,5.070440,0.0001


In [19]:
if PLOT_SUBJECTS:
    subjects_dir = fig_dir_per_iter / "pmin-per-subject"
    subjects_dir.mkdir(exist_ok=True, parents=True)

    for subject_id in subjects_plot_data_per_iter["subject"].unique():
        subject_data = subjects_plot_data_per_iter[
            subjects_plot_data_per_iter["subject"] == subject_id
        ]
        viz_pmin(
            subject_data,
            estimated="estimated_pmin",
            vprec="vprec_pmin",
            thresholds=THRESHOLDS,
            title=f"pmin at each iteration - Subject {subject_id}",
            out_file=subjects_dir / f"{subject_id}.pdf",
            show=False,
        )

## Merging $P_{min}$ and VPREC to compute diff $p_{min}$ at each iteration (Threshold at last iteration)


In [20]:
fig_dir_last_iter = Path("paper", "figures", "pmin-threshold-last-iteration")
fig_dir_last_iter.mkdir(exist_ok=True, parents=True)

In [21]:
# Step 1: extract the Binary64 metric at last iteration per group
vprec_df_ = vprec_df.copy()
ref = (
    vprec_df_[vprec_df_["data_format"] == "binary64"]
    .groupby(["subject", "stage", "level", "iteration"])["metric"]
    .first()  # Only 1 element per group
    .rename("binary64_metric")
    .reset_index()
)
idxmax_iter_fp64 = ref.groupby(["subject", "stage", "level"])["iteration"].idxmax()
ref = ref.loc[idxmax_iter_fp64].rename(
    columns={"binary64_metric": "final_binary64_metric"}
)

# Step 2: join back to the original DataFrame
vprec_df_ = vprec_df_.merge(
    ref.drop(columns="iteration"),
    on=["subject", "stage", "level"],
    how="left",
)

# Step 3: compute the difference
vprec_df_["metric_rel_diff"] = (
    vprec_df_["metric"] - vprec_df_["final_binary64_metric"]
) / np.abs(vprec_df_["final_binary64_metric"])

### Global


In [22]:
# VPREC minimum precision per stage, level, iteration
vprec = (
    vprec_df_.groupby(["exp_id", "precision", "range", "stage", "level", "iteration"])[
        "metric_rel_diff"
    ]
    .mean()
    .rename("mean_rel_diff")
    .reset_index()
)

# Get mean_rel_diff at last iteration
idxmax_iter = vprec.groupby(["exp_id", "precision", "range", "stage", "level"])[
    "iteration"
].idxmax()

last_iter = vprec.loc[
    idxmax_iter,
    ["exp_id", "precision", "range", "stage", "level", "mean_rel_diff"],
].rename(columns={"mean_rel_diff": "final_mean_rel_diff"})

vprec = vprec.merge(
    last_iter,
    on=["exp_id", "precision", "range", "stage", "level"],
)

# PMIN mean estimate per stage, level, iteration, direction
estimated_pmin = (
    pmin_df.groupby(["stage", "level", "iteration", "direction"])["pmin"]
    .mean()
    .rename("estimated_pmin")
    .reset_index()
)


def get_threshold_data(
    *, vprec: pd.DataFrame, estimated_pmin: pd.DataFrame, threshold: float
) -> pd.DataFrame:
    # Threshold based on last iteration
    vprec = vprec[vprec["final_mean_rel_diff"] <= threshold]

    # Retrieve minimum precision per stage, level, iteration
    vprec_pmin = (
        vprec.groupby(["exp_id", "stage", "level", "iteration"])["precision"]
        .min()
        .rename("vprec_pmin")
        .reset_index()
    )

    plot_data = estimated_pmin.merge(
        vprec_pmin, on=["stage", "level", "iteration"], how="left"
    )
    plot_data["diff_pmin"] = plot_data["vprec_pmin"] - plot_data["estimated_pmin"]
    plot_data["threshold"] = threshold
    return plot_data


dfs = list()
for threshold in THRESHOLDS:
    plot_data = get_threshold_data(
        vprec=vprec, estimated_pmin=estimated_pmin, threshold=threshold
    )
    plot_data["threshold"] = threshold
    dfs.append(plot_data)

plot_data_last_iter = pd.concat(dfs, ignore_index=True)
plot_data_last_iter

,stage,level,iteration,direction,estimated_pmin,exp_id,vprec_pmin,diff_pmin,threshold
0,Affine,0,1,nan,8.228812,0000,53,44.771188,0.0100
1,Affine,0,1,nan,8.228812,0001,6,-2.228812,0.0100
2,Affine,0,1,nan,8.228812,0011,7,-1.228812,0.0100
3,Affine,0,1,nan,8.228812,0100,7,-1.228812,0.0100
4,Affine,0,1,nan,8.228812,0110,8,-0.228812,0.0100
...,...,...,...,...,...,...,...,...,...
26114,SyN,3,20,moving,12.538473,0011,15,2.461527,0.0001
26115,SyN,3,20,moving,12.538473,0100,7,-5.538473,0.0001
26116,SyN,3,20,moving,12.538473,0110,7,-5.538473,0.0001
26117,SyN,3,20,moving,12.538473,0111,15,2.461527,0.0001


In [23]:
viz_pmin(
    plot_data_last_iter,
    estimated="estimated_pmin",
    vprec="vprec_pmin",
    thresholds=THRESHOLDS,
    title="pmin at last iteration",
    out_file=fig_dir_last_iter / "pmin-overview.pdf",
)


threshold=1e-02,avg_under=6.15, std_under=6.90, avg_over=50.59, std_over=43.58
threshold=5e-03,avg_under=7.41, std_under=7.03, avg_over=43.32, std_over=35.24
threshold=1e-03,avg_under=11.57, std_under=9.75, avg_over=34.57, std_over=30.70
threshold=1e-04,avg_under=16.79, std_under=11.40, avg_over=26.03, std_over=22.61


In [24]:
viz_pmin(
    plot_data_last_iter,
    estimated="estimated_pmin",
    vprec="vprec_pmin",
    thresholds=THRESHOLDS,
    title="pmin at last iteration",
    out_file=fig_dir_last_iter / "pmin-overview.pdf",
)


threshold=1e-02,avg_under=6.15, std_under=6.90, avg_over=50.59, std_over=43.58
threshold=5e-03,avg_under=7.41, std_under=7.03, avg_over=43.32, std_over=35.24
threshold=1e-03,avg_under=11.57, std_under=9.75, avg_over=34.57, std_over=30.70
threshold=1e-04,avg_under=16.79, std_under=11.40, avg_over=26.03, std_over=22.61


In [25]:
import numpy as np

print(
    "0.5% :", np.mean([15.3, 9.3, 23.4, 1.8, 7.4, 5.7, 0.6, 0.6, 2.0, 15.9, 6.1, 0.9])
)
print(
    "0.1% :", np.mean([22.0, 17.1, 24.4, 2.9, 8.0, 5.9, 1.4, 1.5, 4.2, 28.7, 20.5, 2.4])
)

0.5% : 7.416666666666667
0.1% : 11.583333333333336


### Per subject-level


In [26]:
# Use last iteration per subject to compute VPREC minimum precision
subject_vprec = vprec_df_.copy()
idxmax_iter = subject_vprec.groupby(
    ["subject", "exp_id", "precision", "range", "stage", "level"]
)["iteration"].idxmax()

last_iter = subject_vprec.loc[
    idxmax_iter,
    [
        "subject",
        "exp_id",
        "precision",
        "range",
        "stage",
        "level",
        "metric_rel_diff",
    ],
].rename(columns={"metric_rel_diff": "final_metric_rel_diff"})

subject_vprec = subject_vprec.merge(
    last_iter,
    on=["subject", "exp_id", "precision", "range", "stage", "level"],
)
# PMIN estimate
subjects_pmin_means = (
    pmin_df.groupby(["subject", "stage", "level", "iteration", "direction"])["pmin"]
    .mean()
    .rename("estimated_pmin")
    .reset_index()
)


def get_threshold_data(
    *, vprec: pd.DataFrame, estimated_pmin: pd.DataFrame, threshold: float
) -> pd.DataFrame:
    # Threshold based on last iteration
    vprec = vprec[vprec["final_metric_rel_diff"] <= threshold]

    vprec_pmin = (
        vprec.groupby(["subject", "exp_id", "stage", "level", "iteration"])["precision"]
        .min()
        .rename("vprec_pmin")
        .reset_index()
    )

    plot_data = estimated_pmin.merge(
        vprec_pmin, on=["subject", "stage", "level", "iteration"], how="left"
    )
    plot_data["diff_pmin"] = plot_data["vprec_pmin"] - plot_data["estimated_pmin"]
    plot_data["threshold"] = threshold
    return plot_data


dfs = list()
for threshold in THRESHOLDS:
    subject_plot_data = get_threshold_data(
        vprec=subject_vprec,
        estimated_pmin=subjects_pmin_means,
        threshold=threshold,
    )
    dfs.append(subject_plot_data)

subjects_plot_data_last_iter = pd.concat(dfs, ignore_index=True)
subjects_plot_data_last_iter

,subject,stage,level,iteration,direction,estimated_pmin,exp_id,vprec_pmin,diff_pmin,threshold
0,0003001,Affine,0,1,nan,7.689411,0000,53.0,45.310589,0.0100
1,0003001,Affine,0,1,nan,7.689411,0001,6.0,-1.689411,0.0100
2,0003001,Affine,0,1,nan,7.689411,0011,12.0,4.310589,0.0100
3,0003001,Affine,0,1,nan,7.689411,0100,7.0,-0.689411,0.0100
4,0003001,Affine,0,1,nan,7.689411,0110,11.0,3.310589,0.0100
...,...,...,...,...,...,...,...,...,...,...
855379,0003064,SyN,3,20,moving,12.929560,0011,16.0,3.070440,0.0001
855380,0003064,SyN,3,20,moving,12.929560,0100,7.0,-5.929560,0.0001
855381,0003064,SyN,3,20,moving,12.929560,0110,9.0,-3.929560,0.0001
855382,0003064,SyN,3,20,moving,12.929560,0111,18.0,5.070440,0.0001


In [27]:
if PLOT_SUBJECTS:
    subjects_dir = fig_dir_last_iter / "pmin-per-subject"
    subjects_dir.mkdir(exist_ok=True, parents=True)

    for subject_id in subjects_plot_data_last_iter["subject"].unique():
        subject_data = subjects_plot_data_last_iter[
            subjects_plot_data_last_iter["subject"] == subject_id
        ]
        viz_pmin(
            subject_data,
            estimated="estimated_pmin",
            vprec="vprec_pmin",
            thresholds=THRESHOLDS,
            title=f"pmin at last iteration - Subject {subject_id}",
            out_file=subjects_dir / f"{subject_id}.pdf",
            show=False,
        )

In [28]:
if PLOT_SUBJECTS:
    subjects_dir = fig_dir_last_iter / "pmin-per-subject"
    subjects_dir.mkdir(exist_ok=True, parents=True)

    for subject_id in subjects_plot_data_last_iter["subject"].unique():
        subject_data = subjects_plot_data_last_iter[
            subjects_plot_data_last_iter["subject"] == subject_id
        ]
        viz_pmin(
            subject_data,
            estimated="estimated_pmin",
            vprec="vprec_pmin",
            thresholds=THRESHOLDS,
            title=f"pmin at last iteration - Subject {subject_id}",
            out_file=subjects_dir / f"{subject_id}.pdf",
            show=False,
        )